# 🌦️ Polymarket Weather — Inefficiency Hunter

Compares **weather forecast probabilities** (Gaussian model over Open-Meteo daily forecasts)  
against **Polymarket implied probabilities** to find exploitable edges.

**Data layout expected:**
```
data/
  london_snapshots.csv
  london_daily.csv
  london_hourly.csv
  chicago_snapshots.csv
  ...
```

In [ ]:
# ── Install deps (Colab / fresh env) ──────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'matplotlib', 'scipy', 'seaborn',
                'plotly', 'ipywidgets'], check=False)

In [ ]:
# ── Imports & config ──────────────────────────────────────────────────────────
from pathlib import Path
import sys

# If running from the same dir as the script:
sys.path.insert(0, str(Path('.').resolve()))
from polymarket_weather_analysis import (
    load_snapshots, load_daily, load_hourly,
    analyze_city, forecast_drift_analysis, market_prob_evolution,
    WeatherBettingBot, CITY_ALIASES,
    parse_question, get_forecast_at_time, temp_prob_from_forecast,
    days_ahead,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_DIR   = Path('./data')     # ← change if needed
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Auto-discover cities ──────────────────────────────────────────────────────
CITIES = [f.stem.replace('_snapshots','') for f in DATA_DIR.glob('*_snapshots.csv')]
print('Cities found:', CITIES)

## 1 · Load & Inspect Raw Data

In [ ]:
# Load all cities
snapshots = {c: load_snapshots(DATA_DIR, c) for c in CITIES}
daily     = {c: load_daily(DATA_DIR, c)     for c in CITIES}

# Show snapshot sample
city = CITIES[0]
print(f'--- {city} snapshots ({len(snapshots[city])} rows) ---')
snapshots[city][['question','yes_prob','liquidity_usdc','end_date_iso','fetched_at_utc']].head(10)

In [ ]:
# Daily forecast sample
print(f'--- {city} daily forecast ({len(daily[city])} rows) ---')
daily[city].head(10)

## 2 · Forecast Drift — How Does the Forecast Evolve?

In [ ]:
def plot_drift_interactive(city, n_dates=8):
    drift = forecast_drift_analysis(DATA_DIR, city)
    if drift.empty:
        print('No drift data'); return

    top_dates = sorted(drift['target_date'].unique())[-n_dates:]
    drift_sub = drift[drift['target_date'].isin(top_dates)]

    fig = px.line(
        drift_sub, x='fetched_at', y='temp_max_c',
        color=drift_sub['target_date'].astype(str),
        markers=True,
        title=f'{CITY_ALIASES.get(city, city)} — Max Temp Forecast Drift',
        labels={'fetched_at':'Snapshot UTC','temp_max_c':'Forecast Max °C'},
        template='plotly_dark',
    )
    fig.update_traces(marker_size=4)
    fig.show()

for c in CITIES:
    plot_drift_interactive(c)

## 3 · Market Probability Evolution

In [ ]:
def plot_market_evo_interactive(city, top_n=10):
    mkt = market_prob_evolution(snapshots[city])
    if mkt.empty:
        return

    # Keep only top_n most-updated markets
    top_ids = mkt.groupby('condition_id')['yes_prob'].count().nlargest(top_n).index
    mkt_sub = mkt[mkt['condition_id'].isin(top_ids)]

    fig = px.line(
        mkt_sub, x='fetched_at', y='yes_prob',
        color='condition_id',
        hover_data=['question'],
        title=f'{CITY_ALIASES.get(city, city)} — Market P(Yes) Evolution',
        labels={'fetched_at':'Time UTC','yes_prob':'Market P(Yes)'},
        template='plotly_dark',
        range_y=[-0.05, 1.05],
    )
    fig.update_traces(marker_size=4)
    fig.show()

for c in CITIES:
    plot_market_evo_interactive(c)

## 4 · Opportunity Detection — Forecast vs Market

In [ ]:
MIN_EDGE      = 0.08   # 8 pp edge
MIN_LIQUIDITY = 500    # USDC

all_opps = []
for c in CITIES:
    opps = analyze_city(DATA_DIR, c, min_edge=MIN_EDGE, min_liquidity=MIN_LIQUIDITY)
    if not opps.empty:
        all_opps.append(opps)
        print(f'{c}: {len(opps)} opportunities')

if all_opps:
    combined = pd.concat(all_opps, ignore_index=True)
    print(f'\nTotal: {len(combined)} opportunities')
else:
    combined = pd.DataFrame()
    print('No opportunities found — try lowering MIN_EDGE')

In [ ]:
if not combined.empty:
    top_cols = ['city','target_date','days_ahead','forecast_temp','parsed_temp',
                'condition','forecast_prob','market_prob','edge','bet_side',
                'kelly_fraction','liquidity_usdc']
    display(combined[top_cols].sort_values('edge', ascending=False).head(30))

## 5 · Interactive Scatter: Forecast vs Market Probability

In [ ]:
if not combined.empty:
    fig = px.scatter(
        combined,
        x='market_prob', y='forecast_prob',
        color='bet_side',
        size='edge',
        hover_data=['city','target_date','days_ahead','question','liquidity_usdc'],
        color_discrete_map={'Yes':'#3fb950','No':'#f85149'},
        title='Forecast Probability vs Market Probability',
        labels={'market_prob':'Market P(Yes)','forecast_prob':'Forecast P(Yes)'},
        template='plotly_dark',
        opacity=0.8,
    )
    # Add fair-value line
    fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1,
                  line=dict(color='gray', dash='dash'))
    fig.update_layout(xaxis_range=[-0.05,1.05], yaxis_range=[-0.05,1.05])
    fig.show()

## 6 · Edge vs Forecast Horizon

In [ ]:
if not combined.empty:
    fig = px.scatter(
        combined,
        x='days_ahead', y='edge',
        color='city', symbol='bet_side',
        hover_data=['question','market_prob','forecast_prob'],
        title='Edge vs Days-Ahead Horizon',
        labels={'days_ahead':'Days ahead at time of snapshot','edge':'Edge'},
        template='plotly_dark',
    )
    fig.add_hline(y=MIN_EDGE, line_dash='dash', line_color='yellow',
                  annotation_text=f'Min edge ({MIN_EDGE:.0%})')
    fig.show()

    # Key insight: does edge decay with time?
    print('Correlation (days_ahead, edge):', combined[['days_ahead','edge']].corr().iloc[0,1].round(3))

## 7 · Anomaly Deep-Dive: Large Dislocations

In [ ]:
if not combined.empty:
    threshold_anomaly = 0.20  # 20pp dislocation
    anomalies = combined[combined['edge'] >= threshold_anomaly].copy()
    print(f'Anomalies (edge ≥ {threshold_anomaly:.0%}): {len(anomalies)}')
    if not anomalies.empty:
        display(anomalies.sort_values('edge', ascending=False)[
            ['city','question','target_date','days_ahead',
             'forecast_prob','market_prob','edge','kelly_fraction','liquidity_usdc']
        ])

## 8 · Bot Dry-Run Simulation

In [ ]:
if not combined.empty:
    BANKROLL = 1000.0  # USDC

    bot = WeatherBettingBot(
        min_edge=MIN_EDGE,
        min_liquidity=MIN_LIQUIDITY,
        max_kelly=0.20,
        bankroll_usdc=BANKROLL,
        dry_run=True,
    )
    bot.run(combined)

    # Show bet log
    if bot.bet_log:
        bets_df = pd.DataFrame(bot.bet_log)
        print('\nBet log:')
        display(bets_df[['condition_id','bet_side','size_usdc',
                          'market_prob','forecast_prob','edge']].head(20))
        print(f'\nTotal exposure: ${bets_df["size_usdc"].sum():.2f} / ${BANKROLL:.0f} bankroll')
        print(f'Expected value: ${(bets_df["edge"] * bets_df["size_usdc"]).sum():.2f}')

## 9 · Historical Calibration (Closed Markets)

Check how often our model was right on closed markets where we know the outcome.

In [ ]:
# Closed markets have Yes prob in {0, 1} at resolution
calibration_records = []
for city in CITIES:
    snap = snapshots[city]
    closed = snap[snap.get('closed', pd.Series(False, index=snap.index)) == True].copy()
    if closed.empty:
        # Fall back: markets where yes_prob ∈ {0.0, 1.0} at most recent snapshot
        last = snap.sort_values('fetched_at_utc').groupby('condition_id').last()
        closed = last[(last['yes_prob'].isin([0.0, 1.0]))].reset_index()

    for _, row in closed.iterrows():
        q = str(row.get('question',''))
        parsed = parse_question(q)
        if not parsed:
            continue
        target = row.get('end_date_iso')
        fetch  = row.get('fetched_at_utc')
        if pd.isna(target) or pd.isna(fetch):
            continue
        # The very last yes_prob is the resolution
        resolved_yes = row.get('yes_prob', np.nan)
        if np.isnan(resolved_yes):
            continue

        # Find earliest forecast for this target date
        try:
            d = daily[city]
            fr = get_forecast_at_time(d, target, fetch)
            if fr is None:
                continue
            d_ahead = days_ahead(target, fetch)
            sigma = min(1.0 + 0.4*max(0, d_ahead), 4.0)
            fp = temp_prob_from_forecast(parsed, fr, sigma)
            calibration_records.append({
                'city': city,
                'question': q[:60],
                'resolved_yes': resolved_yes,
                'forecast_prob': fp,
                'correct': (resolved_yes >= 0.5) == (fp >= 0.5),
            })
        except Exception:
            continue

calib = pd.DataFrame(calibration_records)
if not calib.empty:
    acc = calib['correct'].mean()
    print(f'Calibration accuracy: {acc:.1%} over {len(calib)} resolved markets')
    display(calib.head(20))
else:
    print('No resolved markets found in data.')

## 10 · Export Opportunities

In [ ]:
if not combined.empty:
    out = OUTPUT_DIR / 'opportunities.csv'
    combined.sort_values('edge', ascending=False).to_csv(out, index=False)
    print(f'Saved {len(combined)} rows → {out}')